# 13 — Factless Fact Table — DuckDB

Tabela fato sem métricas — registra eventos de cobertura territorial.

In [1]:
import sys, os
sys.path.insert(0, os.getcwd())
from utils import get_conn, DB_PATH, DATA_DIR

conn = get_conn()
print(f"Conectado: {DB_PATH}")

Conectado: /workspace/pf_northwind/duckdb/northwind_dw.duckdb


In [2]:
# ============================================================
# Criar FactEmployeeTerritoryActivity
# ============================================================
conn.execute("""
    CREATE OR REPLACE TABLE gold.FactEmployeeTerritoryActivity AS
    SELECT DISTINCT
        CAST(strftime(o.OrderDate, '%Y%m%d') AS INTEGER) AS ActivityDateKey,
        de.EmployeeSK,
        b.TerritorySK
    FROM bronze.orders o
    JOIN gold.DimEmployee              de ON de.EmployeeID = o.EmployeeID
    JOIN gold.BridgeEmployeeTerritory  b  ON b.EmployeeSK = de.EmployeeSK
""")

n = conn.execute("SELECT COUNT(*) AS n FROM gold.FactEmployeeTerritoryActivity").fetchdf()['n'][0]
print(f"FactEmployeeTerritoryActivity: {n} eventos")
assert n > 0, "Factless fact vazia!"
print("✓ OK")


FactEmployeeTerritoryActivity: 3696 eventos
✓ OK


In [3]:
# ============================================================
# DEMO 1: Territórios cobertos vs total
# ============================================================
conn.execute("""
    SELECT
        COUNT(DISTINCT TerritorySK)  AS TerritoriosCobertos,
        (SELECT COUNT(*) FROM gold.DimTerritory) AS TerritoriostTotal
    FROM gold.FactEmployeeTerritoryActivity
""").fetchdf()


,TerritoriosCobertos,TerritoriostTotal
0,49,53


In [4]:
# ============================================================
# DEMO 2: Cobertura por ano e região
# ============================================================
conn.execute("""
    SELECT YEAR(d.FullDate) AS Year,
           dt.RegionName,
           COUNT(DISTINCT fa.TerritorySK) AS TerritoriosCobertos,
           COUNT(*) AS TotalEventos
    FROM gold.FactEmployeeTerritoryActivity fa
    JOIN gold.DimDate d ON d.DateKey = fa.ActivityDateKey
    JOIN gold.DimTerritory dt ON dt.TerritorySK = fa.TerritorySK
    GROUP BY YEAR(d.FullDate), dt.RegionName
    ORDER BY 1, 2
""").fetchdf()


,Year,RegionName,TerritoriosCobertos,TotalEventos
0,1996,Eastern ...,19,327
1,1996,Northern ...,11,111
2,1996,Southern ...,4,72
3,1996,Western ...,15,185
4,1997,Eastern ...,19,734
5,1997,Northern ...,11,341
6,1997,Southern ...,4,264
7,1997,Western ...,15,500
8,1998,Eastern ...,19,494
9,1998,Northern ...,11,219


In [5]:
# ============================================================
# DEMO 3: "O que NÃO aconteceu?" — territórios sem cobertura
# ============================================================
conn.execute("""
    SELECT dt.RegionName, dt.TerritoryDescription,
           de.FullName AS EmpregadoResponsavel
    FROM gold.BridgeEmployeeTerritory b
    JOIN gold.DimTerritory dt ON dt.TerritorySK = b.TerritorySK
    JOIN gold.DimEmployee de ON de.EmployeeSK = b.EmployeeSK
    WHERE NOT EXISTS (
        SELECT 1 FROM gold.FactEmployeeTerritoryActivity fa
        WHERE fa.EmployeeSK = b.EmployeeSK AND fa.TerritorySK = b.TerritorySK
    )
    ORDER BY dt.RegionName
""").fetchdf()


,RegionName,TerritoryDescription,EmpregadoResponsavel
